# Lab 21 — BONUS Challenges (B1 · B3 · B4 · B5)

> **Yêu cầu**: Đã chạy xong NB1→NB5 (core pipeline). Notebook này **dùng lại** adapter và data từ core run.

| Bonus | Điểm | Nội dung | Ước tính |
|---|---|---|---|
| **B1** | +3 | Merge adapter + assert không tụt + hot-swap ≥2 | ~15 phút |
| **B3** | +4 | Reasoning-trace collapse: so `assistant-only` vs `response-only` | ~60 phút |
| **B4** | +3 | Quét rank r∈{8,16,64} cố định `text-linear` | ~45 phút |
| **B5** | +2 | Push adapter lên HuggingFace Hub | ~5 phút |

**Thứ tự chạy**: B1 → B4 → B3 → B5 (B5 sau cùng khi đã có kết quả)

Runtime > Change runtime type > **T4 GPU** trước khi bắt đầu.

In [ ]:
# @title ⚙️ 0. Setup — clone repo của BẠN + install
import os, subprocess, sys

# ✏️ ĐỔI thành repo của bạn
YOUR_REPO = "https://github.com/binhdinhvan/Day21-Track3-DinhVanBinh-2A202602830-Finetuning-Lab.git"  # @param {type:"string"}
REPO_NAME = YOUR_REPO.rstrip("/").split("/")[-1].replace(".git", "")

if not os.path.exists(REPO_NAME):
    subprocess.run(["git", "clone", "-q", YOUR_REPO], check=True)
os.chdir(f"/content/{REPO_NAME}")
subprocess.run(["git", "pull", "-q"], check=False)
sys.path.insert(0, "src")

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"],
               check=True)

import torch
print("repo   :", REPO_NAME)
print("commit :", subprocess.run(["git","rev-parse","--short","HEAD"],
                                 capture_output=True, text=True).stdout.strip())
print("GPU    :", torch.cuda.get_device_name(0) if torch.cuda.is_available()
      else "NONE — Runtime > Change runtime type > T4 GPU")
if torch.cuda.is_available():
    print("VRAM   : %.1f GB" % (torch.cuda.get_device_properties(0).total_memory/1024**3))

In [ ]:
# @title ⚙️ 0b. Kiểm tra adapter và results từ core run
import json, pathlib

ROOT = pathlib.Path("/content") / REPO_NAME
adapter_dir = ROOT / "adapters" / "correct"
results_dir = ROOT / "results"

assert adapter_dir.exists(), f"❌ Chưa có adapters/correct/ — chạy NB3 trước!"
assert (results_dir / "runs.csv").exists(), "❌ Chưa có results/runs.csv — chạy NB3+NB4 trước!"
assert (results_dir / "baselines_frozen.json").exists(), "❌ Chưa có baselines_frozen.json — chạy NB2 trước!"
assert (results_dir / "verdict.json").exists(), "❌ Chưa có verdict.json — chạy NB5 trước!"

print("✅ adapters/correct/ :", list(adapter_dir.iterdir()))
print("✅ results/          :", [f.name for f in results_dir.iterdir() if not f.name.startswith('.')])

verdict = json.loads((results_dir / "verdict.json").read_text())
print("\nverdist (core):", verdict["verdict"])

---
## 🎯 B1 — Merge & phục vụ nhiều adapter (+3 điểm)

> Deck §23. Merge cho overhead suy luận bằng 0 — nhưng bạn mất gì?
> Assert: điểm sau merge không tụt quá 0.01

In [ ]:
# @title B1 — §1: Đo điểm TRƯỚC merge
import json, os, pathlib, sys, time
sys.path.insert(0, str(ROOT / "src"))

os.environ["COMPUTE_TIER"] = "T4"
from labkit import evaluate as ev, generate, report
from labkit.config import get_tier
from peft import PeftModel

TIER = get_tier("T4")

def load_jsonl(p):
    return [json.loads(l) for l in open(p, encoding="utf-8") if l.strip()]

target = load_jsonl(ROOT / "data" / "eval_target.jsonl")
print(f"Đánh giá trên {len(target)} mẫu target")

model, tok = generate.load_base(TIER)
model = PeftModel.from_pretrained(model, str(ROOT / "adapters" / "correct"))
model.eval()

preds, lat = generate.generate_batch(model, tok, [r["input"] for r in target],
                                     system=generate.NAIVE_PROMPT, label="before_merge")
before = sum(ev.triage_field_accuracy(p, r["label"]) for p, r in zip(preds, target)) / len(target)
print(f"\n📊 Điểm TRƯỚC merge: {before:.4f}")

In [ ]:
# @title B1 — §2: Merge và assert không tụt
print("🔀 Đang merge adapter vào base model...")
merged = model.merge_and_unload()

preds_m, _ = generate.generate_batch(merged, tok, [r["input"] for r in target],
                                     system=generate.NAIVE_PROMPT, label="after_merge")
after = sum(ev.triage_field_accuracy(p, r["label"]) for p, r in zip(preds_m, target)) / len(target)
delta = after - before

print(f"📊 Trước merge: {before:.4f}")
print(f"📊 Sau merge:   {after:.4f}   (Δ {delta:+.4f})")

TOL = 0.01
assert delta >= -TOL, (
    f"❌ Điểm TỤT {abs(delta):.4f} sau merge (ngưỡng {TOL}). "
    "Kiểm tra dtype lúc merge; với DoRA cần PEFT ≥ 0.10."
)
print(f"\n✅ PASSED: điểm không tụt quá ngưỡng {TOL}")

# Lưu merge_check.json
merge_check = {"before_merge": before, "after_merge": after,
               "delta": delta, "tolerance": TOL, "n": len(target)}
report.write_json(merge_check, "merge_check.json", results_dir=ROOT / "results")
print("💾 Lưu results/merge_check.json")
print(json.dumps(merge_check, indent=2))

# Giải phóng bộ nhớ
del merged
generate.free_memory()

In [ ]:
# @title B1 — §3: Hot-swap ≥2 adapter trên cùng một base
# Deck §23: base nằm trong VRAM một lần, nhiều adapter chọn theo request
print("🔄 Nạp base + hot-swap nhiều adapter...")

model_swap, tok = generate.load_base(TIER)
model_swap = PeftModel.from_pretrained(model_swap, str(ROOT / "adapters" / "correct"),
                                        adapter_name="correct")
loaded_adapters = ["correct"]

# Load thêm các adapter từ NB4 nếu tồn tại
for extra in ("attn_only", "qlora", "wrong_lr"):
    d = ROOT / "adapters" / extra
    if d.exists() and (d / "adapter_config.json").exists():
        try:
            model_swap.load_adapter(str(d), adapter_name=extra)
            loaded_adapters.append(extra)
            print(f"  ✅ nạp adapter: {extra}")
        except Exception as e:
            print(f"  ⚠ không nạp được {extra}: {e}")

print(f"\n🔌 Adapter đang nạp: {loaded_adapters}")
assert len(loaded_adapters) >= 2, "❌ Cần ≥2 adapter. Kiểm tra NB4 đã chạy chưa."

# Chạy thử hot-swap
ticket = target[0]["input"]
print(f"\n📝 Ticket thử nghiệm: {ticket[:80]}...")
print("-" * 60)

swap_results = {}
for name in loaded_adapters:
    model_swap.set_adapter(name)
    out, lat = generate.generate_batch(model_swap, tok, [ticket],
                                       system=generate.NAIVE_PROMPT)
    swap_results[name] = {"output": out[0][:200], "latency_ms": round(lat, 1)}
    print(f"\n[{name}] → {out[0][:120]}")

print("\n✅ B1 HOÀN THÀNH: merge xong + hot-swap", len(loaded_adapters), "adapter")
del model_swap
generate.free_memory()

### 📝 B1 — Câu hỏi phân tích

**Merge cho overhead suy luận bằng 0, nhưng bạn mất gì?**

Sau merge, adapter được hấp thụ vĩnh viễn vào trọng số base: bạn **mất khả năng hot-swap** và không thể tắt adapter cho từng request. Nếu cần phục vụ nhiều tác vụ từ cùng một base (multi-tenant serving), phải lưu riêng từng checkpoint merged — tốn bộ nhớ đĩa theo bậc O(n_adapters). Ngược lại, giữ adapter riêng cho phép tải một base + N adapter nhỏ, tiết kiệm VRAM đáng kể.

**Khi nào nên giữ adapter riêng?**  
Khi triển khai multi-tenant (nhiều khách hàng/tác vụ dùng chung một GPU), khi cần A/B test adapter, hoặc khi adapter cần được cập nhật thường xuyên mà không muốn re-deploy toàn bộ model.

---
## 🎯 B4 — Quét rank có kiểm soát (+3 điểm)

> Deck §11. **Cố định** `target_modules="text-linear"`, chỉ quét `r ∈ {8, 16, 64}`.
> Câu hỏi: rank có phải đòn bẩy không? So biên độ với thay đổi vị trí và LR.

In [ ]:
# @title B4 — Train 3 run với r=8, r=16, r=64 (text-linear, cố định)
import json, os, pathlib, sys, time
sys.path.insert(0, str(ROOT / "src"))

from datasets import Dataset
from peft import LoraConfig
from trl import SFTConfig, SFTTrainer

from labkit import data, generate, modeling, report, train
from labkit.config import SPECS, get_tier, training_epochs

TIER = get_tier("T4")
MASK_MODE = os.environ.get("MASK_MODE", "assistant-only")

def load_jsonl(p):
    return [json.loads(l) for l in open(p, encoding="utf-8") if l.strip()]

train_rows = load_jsonl(ROOT / "data" / "split" / "train.jsonl")
target_rows = load_jsonl(ROOT / "data" / "eval_target.jsonl")

# Dùng cùng config base như `correct`, chỉ thay rank
BASE_SPEC = SPECS["correct"]   # text-linear, LR=1e-4, fp16
RANKS_TO_SWEEP = [8, 16, 64]   # r=16 là correct run, dùng làm anchor

from labkit import evaluate as ev

def train_and_score_rank(r: int) -> dict:
    """Train one rank variant and return scores."""
    run_key = f"rank_sweep_r{r}"
    print(f"\n{'='*50}")
    print(f"🚀 Training rank r={r} (text-linear, LR={BASE_SPEC.lr})")
    print(f"{'='*50}")

    model, tok = generate.load_base(TIER, load_in_4bit=False)
    targets_mods = modeling.resolve_target_modules(model, "text-linear")

    # Count params for this rank
    trainable = modeling.count_lora_params(model, targets_mods, r)
    print(f"  trainable params @ r={r}: {trainable/1e6:.2f}M")

    # Build dataset
    rows = data.to_training_dataset(tok, train_rows, max_length=TIER.max_length,
                                    mask_mode=MASK_MODE)
    train_ds = Dataset.from_list(rows)

    EPOCHS = training_epochs()
    STEPS = train.planned_steps(len(rows), TIER, EPOCHS)

    # Use correct spec but override rank
    import dataclasses
    spec_r = dataclasses.replace(BASE_SPEC, r=r, alpha=2*r, key=run_key)

    want_sft = train.sft_config_kwargs(
        TIER, spec_r,
        output_dir=str(ROOT / "adapters" / run_key),
        num_train_epochs=EPOCHS,
        mask_mode=MASK_MODE,
        total_steps=STEPS,
    )
    sft_kwargs, _ = train.filter_kwargs(SFTConfig, want_sft, label="SFTConfig")

    want_lora = train.lora_config_kwargs(spec_r, targets_mods)
    lora_kwargs, _ = train.filter_kwargs(LoraConfig, want_lora, label="LoraConfig")

    generate.free_memory()
    trainer = SFTTrainer(
        model=model,
        args=SFTConfig(**sft_kwargs),
        train_dataset=train_ds,
        processing_class=tok,
        peft_config=LoraConfig(**lora_kwargs),
    )
    fix = train.align_trainable_precision(trainer.model)

    t0 = time.perf_counter()
    result = trainer.train()
    elapsed = time.perf_counter() - t0
    final_loss = result.training_loss
    vram = generate.peak_vram_gb()
    print(f"  train {elapsed:.0f}s  loss={final_loss:.4f}  VRAM={vram:.2f}GB")

    # Save adapter
    out_dir = ROOT / "adapters" / run_key
    trainer.model.save_pretrained(out_dir)
    tok.save_pretrained(out_dir)

    # Quick eval on target
    from peft import PeftModel as PM
    eval_model = trainer.model
    eval_model.eval()
    preds, lat = generate.generate_batch(
        eval_model, tok, [r_["input"] for r_ in target_rows],
        system=generate.NAIVE_PROMPT, label=f"{run_key}/target")
    target_score = sum(ev.triage_field_accuracy(p, r_["label"])
                       for p, r_ in zip(preds, target_rows)) / len(target_rows)
    fmt_score = sum(ev.has_required_keys(p, ev.TRIAGE_KEYS) for p in preds) / len(preds)

    row = {
        "run": run_key, "r": r, "placement": "text-linear",
        "trainable_params": trainable, "learning_rate": BASE_SPEC.lr,
        "final_loss": round(final_loss, 4),
        "target_score": round(target_score, 4),
        "format_score": round(fmt_score, 4),
        "train_seconds": round(elapsed, 1),
        "peak_vram_gb": round(vram, 2),
        "max_steps": STEPS,
    }
    print(f"  target={target_score:.4f}  format={fmt_score:.4f}")

    del eval_model
    generate.free_memory()
    return row

# Run sweep
b4_results = []
for rank in RANKS_TO_SWEEP:
    row = train_and_score_rank(rank)
    b4_results.append(row)

# Save B4 results
report.write_json(b4_results, "b4_rank_sweep.json", results_dir=ROOT / "results")
print("\n💾 Lưu results/b4_rank_sweep.json")

In [ ]:
# @title B4 — Phân tích kết quả rank sweep
import json

b4 = json.loads((ROOT / "results" / "b4_rank_sweep.json").read_text())

print("=" * 70)
print("B4 — Rank Sweep Results (text-linear fixed, same LR & steps)")
print("=" * 70)
print(f"{'rank':>6} | {'params (M)':>10} | {'train_loss':>10} | {'target':>8} | {'VRAM':>6} | {'time(s)':>8}")
print("-" * 70)
for row in b4:
    print(f"{row['r']:>6} | {row['trainable_params']/1e6:>10.2f} | "
          f"{row['final_loss']:>10.4f} | {row['target_score']:>8.4f} | "
          f"{row['peak_vram_gb']:>6.2f} | {row['train_seconds']:>8.1f}")

# Compare amplitude of rank change vs position change vs LR change
runs_csv_path = ROOT / "results" / "runs.csv"
import csv
core_runs = {}
with open(runs_csv_path) as f:
    for row in csv.DictReader(f):
        core_runs[row["run"]] = row

print("\n" + "=" * 70)
print("So sánh 3 'nút vặn': rank vs vị trí vs LR")
print("=" * 70)

# Get target scores from autopsy.json
autopsy = json.loads((ROOT / "results" / "autopsy.json").read_text())
autopsy_d = {r["run"]: r for r in autopsy}

r8  = next((r for r in b4 if r["r"] == 8), None)
r16 = next((r for r in b4 if r["r"] == 16), None)
r64 = next((r for r in b4 if r["r"] == 64), None)

if r8 and r64:
    rank_delta = r64["target_score"] - r8["target_score"]
    print(f"  Thay đổi rank (r=8→64):        target Δ = {rank_delta:+.4f}")

if "correct" in autopsy_d and "attn_only" in autopsy_d:
    pos_delta = autopsy_d["attn_only"]["target"] - autopsy_d["correct"]["target"]
    print(f"  Thay đổi vị trí (attn vs all):  target Δ = {pos_delta:+.4f}")

if "correct" in autopsy_d and "wrong_lr" in autopsy_d:
    lr_delta = autopsy_d["wrong_lr"]["target"] - autopsy_d["correct"]["target"]
    print(f"  Thay đổi LR (1e-4→1e-5):        target Δ = {lr_delta:+.4f}")

print("\n✅ B4 PHÂN TÍCH XONG")

### 📝 B4 — Phân tích: Rank có phải đòn bẩy không?

> **Điền vào bảng sau sau khi chạy xong:**

| Nút vặn | Thay đổi | target Δ | Xếp hạng ảnh hưởng |
|---|---|---|---|
| Rank | r=8 → r=64 | ___ | |
| Vị trí | attn_only → all-linear | ___ | |
| LR | 1e-5 → 1e-4 | ___ | |

**Deck §11 dự đoán**: Rank là *năng lực so với lượng thông tin trong dữ liệu*, không phải nút chỉnh chất lượng. Với 250 mẫu huấn luyện, dữ liệu có đủ thông tin để r=64 dùng hết không?

---
## 🎯 B3 — Reasoning-trace collapse (+4 điểm) ⭐

> Deck §17.5. Fine-tune model biết suy luận bằng dữ liệu Q&A thông thường **phá huỷ
> năng lực suy luận trong khi accuracy vẫn tăng**.
>
> Train 2 lần: `MASK_MODE=assistant-only` vs `MASK_MODE=response-only`, so sánh `valid_trace_rate`.

In [ ]:
# @title B3 — Helper: đo valid_trace_rate
import re

THINK_OPEN  = re.compile(r'<think>', re.IGNORECASE)
THINK_CLOSE = re.compile(r'</think>', re.IGNORECASE)
THINK_BODY  = re.compile(r'<think>(.+?)</think>', re.DOTALL | re.IGNORECASE)

def valid_trace_rate(predictions: list[str]) -> dict:
    """
    Đếm tỷ lệ output có cặp <think>...</think> với nội dung không rỗng.
    
    valid  = có cả open + close + body dài > 5 ký tự
    empty  = có <think></think> nhưng body rỗng (collapse!)
    absent = không có <think> nào
    """
    n = len(predictions)
    valid = empty = absent = 0
    for p in predictions:
        m = THINK_BODY.search(p)
        if m and len(m.group(1).strip()) > 5:
            valid += 1
        elif THINK_OPEN.search(p):
            empty += 1   # <think> present but body collapsed
        else:
            absent += 1
    return {
        "n": n,
        "valid_trace_rate": round(valid / n, 4),
        "empty_trace_rate": round(empty / n, 4),
        "absent_rate":      round(absent / n, 4),
    }

print("✅ valid_trace_rate helper loaded")

In [ ]:
# @title B3 — Run 1: MASK_MODE=assistant-only (default)
# assistant-only: tính loss trên TOÀN BỘ assistant turn (kể cả <think>...)
# → model học cả phong cách suy luận, nhưng có thể collapse trace
import os, json, time
from datasets import Dataset
from peft import LoraConfig
from trl import SFTConfig, SFTTrainer
from labkit import data, evaluate as ev, generate, modeling, report, train
from labkit.config import SPECS, get_tier, training_epochs

TIER = get_tier("T4")

def train_b3(mask_mode: str) -> dict:
    run_key = f"b3_{mask_mode.replace('-', '_')}"
    print(f"\n{'='*55}")
    print(f"🧠 B3 Training: MASK_MODE={mask_mode}")
    print(f"{'='*55}")

    train_rows = load_jsonl(ROOT / "data" / "split" / "train.jsonl")
    target_rows = load_jsonl(ROOT / "data" / "eval_target.jsonl")
    SPEC = SPECS["correct"]

    model, tok = generate.load_base(TIER, load_in_4bit=False)
    targets_mods = modeling.resolve_target_modules(model, SPEC.target)
    trainable = modeling.count_lora_params(model, targets_mods, SPEC.r)

    rows = data.to_training_dataset(tok, train_rows, max_length=TIER.max_length,
                                    mask_mode=mask_mode)
    sup = sum(sum(1 for x in r["labels"] if x != data.IGNORE_INDEX) for r in rows)
    tot = sum(len(r["labels"]) for r in rows)
    print(f"  mask_mode={mask_mode} → supervised {sup}/{tot} ({sup/tot:.1%})")

    train_ds = Dataset.from_list(rows)
    EPOCHS = training_epochs()
    STEPS = train.planned_steps(len(rows), TIER, EPOCHS)

    import dataclasses
    spec_r = dataclasses.replace(SPEC, key=run_key)
    want_sft = train.sft_config_kwargs(
        TIER, spec_r,
        output_dir=str(ROOT / "adapters" / run_key),
        num_train_epochs=EPOCHS,
        mask_mode=mask_mode,
        total_steps=STEPS,
    )
    sft_kwargs, _ = train.filter_kwargs(SFTConfig, want_sft, label="SFTConfig")
    want_lora = train.lora_config_kwargs(spec_r, targets_mods)
    lora_kwargs, _ = train.filter_kwargs(LoraConfig, want_lora, label="LoraConfig")

    generate.free_memory()
    trainer = SFTTrainer(
        model=model,
        args=SFTConfig(**sft_kwargs),
        train_dataset=train_ds,
        processing_class=tok,
        peft_config=LoraConfig(**lora_kwargs),
    )
    train.align_trainable_precision(trainer.model)

    t0 = time.perf_counter()
    result = trainer.train()
    elapsed = time.perf_counter() - t0
    print(f"  train {elapsed:.0f}s  loss={result.training_loss:.4f}")

    # Save adapter
    out_dir = ROOT / "adapters" / run_key
    trainer.model.save_pretrained(out_dir)
    tok.save_pretrained(out_dir)

    # Eval: target score + valid_trace_rate
    trainer.model.eval()
    preds, lat = generate.generate_batch(
        trainer.model, tok, [r["input"] for r in target_rows],
        system=generate.NAIVE_PROMPT, label=f"{run_key}/target")
    target_score = sum(ev.triage_field_accuracy(p, r["label"])
                       for p, r in zip(preds, target_rows)) / len(target_rows)

    # Đo valid_trace_rate: dùng câu hỏi mở để model có cơ hội sinh <think>
    trace_prompts = [
        "Hãy tính 15% của 240.",
        "Giải thích ngắn gọn vì sao bầu trời màu xanh?",
        "Nếu A > B và B > C, thì A so với C như thế nào?",
        "Cho dãy số 2, 4, 8, 16, số tiếp theo là bao nhiêu?",
        "Từ nào là từ trái nghĩa của 'nhanh'?",
    ] * 10  # 50 câu
    trace_preds, _ = generate.generate_batch(
        trainer.model, tok, trace_prompts[:50],
        system=None, max_new_tokens=256, label=f"{run_key}/trace")
    trace_stats = valid_trace_rate(trace_preds)

    row = {
        "run": run_key,
        "mask_mode": mask_mode,
        "target_score": round(target_score, 4),
        "valid_trace_rate": trace_stats["valid_trace_rate"],
        "empty_trace_rate": trace_stats["empty_trace_rate"],
        "absent_rate": trace_stats["absent_rate"],
        "final_loss": round(result.training_loss, 4),
        "train_seconds": round(elapsed, 1),
    }
    print(f"\n  target={target_score:.4f}  valid_trace_rate={trace_stats['valid_trace_rate']:.4f}")
    print(f"  empty_trace={trace_stats['empty_trace_rate']:.4f}  absent={trace_stats['absent_rate']:.4f}")

    del trainer
    generate.free_memory()
    return row

# Train run 1: assistant-only
b3_row1 = train_b3("assistant-only")
print("\n✅ B3 Run 1 (assistant-only) xong!")

In [ ]:
# @title B3 — Run 2: MASK_MODE=response-only
# response-only: chỉ tính loss trên JSON output SAU </think>
# → model KHÔNG học phong cách suy luận, trace có thể collapse
b3_row2 = train_b3("response-only")
print("\n✅ B3 Run 2 (response-only) xong!")

In [ ]:
# @title B3 — Tổng hợp kết quả + lưu file
b3_results = [b3_row1, b3_row2]
report.write_json(b3_results, "b3_trace_collapse.json", results_dir=ROOT / "results")

print("=" * 65)
print("B3 — Reasoning-Trace Collapse Results")
print("=" * 65)
print(f"{'MASK_MODE':>20} | {'target':>8} | {'valid_trace':>11} | {'empty_trace':>11} | {'absent':>8}")
print("-" * 65)
for r in b3_results:
    print(f"{r['mask_mode']:>20} | {r['target_score']:>8.4f} | "
          f"{r['valid_trace_rate']:>11.4f} | {r['empty_trace_rate']:>11.4f} | {r['absent_rate']:>8.4f}")

print("\n💾 Lưu results/b3_trace_collapse.json")
print("✅ B3 HOÀN THÀNH")

### 📝 B3 — Điền kết quả vào bảng:

| MASK_MODE | target | valid_trace_rate | regression |
|---|---|---|---|
| assistant-only | ___ | ___ | ___ |
| response-only  | ___ | ___ | ___ |

**Câu hỏi**: `target` có tăng trong khi `valid_trace_rate` giảm không? Nếu chỉ nhìn `target`, bạn có phát hiện ra vấn đề không? Đây chính là lý do deck §21 nói accuracy một mình không phải bằng chứng.

---
## 🎯 B5 — Push adapter lên HuggingFace Hub (+2 điểm)

> Adapter công khai = kiểm chứng được + đẹp trên CV.

In [ ]:
# @title B5 — Push adapter lên HuggingFace Hub
import subprocess, sys

# ✏️ Điền HuggingFace token và repo name của bạn
HF_TOKEN   = ""  # @param {type:"string"}
HF_REPO_ID = "binhdinhvan/lab21-qwen35-triage-vi"  # @param {type:"string"}

assert HF_TOKEN, "❌ Điền HF_TOKEN vào ô trên (lấy tại https://huggingface.co/settings/tokens)"

from huggingface_hub import HfApi
from peft import PeftModel
from labkit import generate
from labkit.config import get_tier

TIER = get_tier("T4")
adapter_path = ROOT / "adapters" / "correct"

print(f"📤 Push adapter → {HF_REPO_ID}")

# Load model + adapter
model, tok = generate.load_base(TIER)
model = PeftModel.from_pretrained(model, str(adapter_path))

# Push to Hub
model.push_to_hub(
    HF_REPO_ID,
    token=HF_TOKEN,
    commit_message="Lab21: LoRA adapter for Vietnamese CSKH triage (Qwen3.5-4B, r=16, text-linear)",
    private=False,
)
tok.push_to_hub(HF_REPO_ID, token=HF_TOKEN)

HF_LINK = f"https://huggingface.co/{HF_REPO_ID}"
print(f"\n✅ B5 XONG! Adapter public tại:")
print(f"   {HF_LINK}")
print(f"\n📋 Copy link này vào REPORT.md phần Phụ lục → B5")

# Save link to file for reference
with open(ROOT / "results" / "b5_hf_link.txt", "w") as f:
    f.write(HF_LINK + "\n")

del model
generate.free_memory()

---
## 📦 Download kết quả về máy

Sau khi chạy xong, download các file kết quả để cập nhật vào repo.

In [ ]:
# @title 📥 Download tất cả results về máy
from google.colab import files
import json, pathlib, zipfile, io

ROOT = pathlib.Path("/content") / REPO_NAME

# Tạo zip chứa tất cả bonus results
bonus_files = [
    "results/merge_check.json",
    "results/b4_rank_sweep.json",
    "results/b3_trace_collapse.json",
    "results/b5_hf_link.txt",
]

zip_buffer = io.BytesIO()
with zipfile.ZipFile(zip_buffer, 'w', zipfile.ZIP_DEFLATED) as zf:
    for rel_path in bonus_files:
        full_path = ROOT / rel_path
        if full_path.exists():
            zf.write(full_path, rel_path)
            print(f"✅ Added: {rel_path}")
        else:
            print(f"⚠ Không tìm thấy: {rel_path} (chưa chạy bonus này?)")

zip_buffer.seek(0)
with open("/content/bonus_results.zip", "wb") as f:
    f.write(zip_buffer.getvalue())

files.download("/content/bonus_results.zip")
print("\n📥 Đang download bonus_results.zip...")
print("Sau khi download, copy các file JSON vào thư mục results/ của repo local")
print("rồi chạy: git add results/ && git commit -m 'feat: add bonus results' && git push")